In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_groq import ChatGroq

In [8]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [12]:
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0)
def generate_joke(state: JokeState):
    prompt = f"Generate a joke on the topic: {state['topic']}"
    response = llm.invoke(prompt).content
    return {"joke": response}

def generate_explanation(state: JokeState):
    prompt = f"Right an explanation for the joke: {state['joke']}"
    response = llm.invoke(prompt).content
    return {"explanation": response}

In [13]:
# Create the state graph
graph = StateGraph(JokeState)

# Add nodes
graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

# Add edges
graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)

# Instantiate the checkpointer for persistence
checkpointer = MemorySaver()

# Compile graph with the checkpointer registered
workflow = graph.compile(checkpointer=checkpointer)

In [14]:
# Configuration block carrying the specific session identifier
config = {"configurable": {"thread_id": "1"}}

# Initial invocation pass
initial_state = {"topic": "pizza"}
result = workflow.invoke(initial_state, config=config)

In [16]:
print(result)

{'topic': 'pizza', 'joke': "Here's a pizza joke:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!", 'explanation': 'The joke!\n\nThe joke is a play on words. The phrase "feeling crusty" has a double meaning here.\n\nIn one sense, a pizza has a crust, which is the outer layer of the dough that forms the base of the pizza.\n\nBut the phrase "feeling crusty" also sounds similar to "feeling cranky" or "feeling grumpy", which means being in a bad mood.\n\nSo, the joke is saying that the pizza is in a bad mood (crusty as in grumpy), and it\'s making a pun on the fact that a pizza literally has a crust. It\'s a lighthearted and cheesy (pun intended) joke that\'s meant to make you laugh!'}
